# Gene Assignment: ALDH2
Mapping protein coordinates to genome using Ensembl DB (ensembldb + AnnotationHub).

In [13]:
# ============================================================
# CYP2E1 (ENSG00000130649) — Full Pipeline
# Transcripts, Proteins & Protein-to-Genome Mapping
# Based on: ensembldb + AnnotationHub (Bioconductor 3.23)
# ============================================================

# ---------------------------------------------------------
# 1. INSTALL PACKAGES (run once)
# ---------------------------------------------------------
if (!require("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}
BiocManager::install("ensembldb")
BiocManager::install("AnnotationHub")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Warning message:
"package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'ensembldb'"
'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Warning message:
"package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'AnnotationHub'"


In [14]:
# ---------------------------------------------------------
# 2. SETUP & DIRECTORIES
# ---------------------------------------------------------
dir.create("/content/compbioass", recursive = TRUE)
setwd("/content/compbioass")

library(ensembldb)
library(AnnotationHub)

# Set AnnotationHub cache directory
dir.create("/content/compbioass/AH_cache", recursive = TRUE)
setAnnotationHubOption("CACHE", "/content/compbioass/AH_cache")

# Load the same Ensembl database used in class
ah  <- AnnotationHub()
edb <- ah[["AH119325"]]

print("Environment successfully set up!")

Warning message in dir.create("/content/compbioass", recursive = TRUE):
"'/content/compbioass' already exists"
Warning message in dir.create("/content/compbioass/AH_cache", recursive = TRUE):
"'/content/compbioass/AH_cache' already exists"
loading from cache



[1] "Environment successfully set up!"


In [15]:
# ---------------------------------------------------------
# 3. DEFINE YOUR GENE
# ---------------------------------------------------------
my_gene <- "CYP2E1"   # Your assigned gene

In [16]:
# ---------------------------------------------------------
# 4. FETCH TRANSCRIPTS
# ---------------------------------------------------------
gene_transcripts <- transcripts(edb, filter = ~ gene_name == my_gene)

transcript_ids <- gene_transcripts$tx_id
cat("\n--- List of Transcripts for", my_gene, "---\n")
print(transcript_ids)
cat("Total transcripts:", length(transcript_ids), "\n")


--- List of Transcripts for CYP2E1 ---
 [1] "ENST00000463117" "ENST00000541261" "ENST00000252945" "ENST00000477500"
 [5] "ENST00000418356" "ENST00000421586" "ENST00000541080" "ENST00000480558"
 [9] "ENST00000368520" "ENST00000469258"
Total transcripts: 10 


In [17]:
# ---------------------------------------------------------
# 5. FETCH PROTEINS
# ---------------------------------------------------------
gene_proteins <- proteins(edb, filter = ~ gene_name == my_gene)
protein_ids   <- na.omit(gene_proteins$protein_id)

cat("\n--- List of Proteins for", my_gene, "---\n")
print(protein_ids)
cat("Total proteins:", length(protein_ids), "\n")


--- List of Proteins for CYP2E1 ---
[1] "ENSP00000440689" "ENSP00000437799" "ENSP00000252945" "ENSP00000397299"
[5] "ENSP00000412754" "ENSP00000444958"
Total proteins: 6 


In [18]:
# ---------------------------------------------------------
# 6. PROTEIN-TO-GENOME MAPPING LOOP
# ---------------------------------------------------------
mapping_results <- list()

for (pid in protein_ids) {

  # Get the protein sequence length
  prot_info   <- proteins(edb, filter = ~ protein_id == pid)
  prot_length <- nchar(prot_info$protein_sequence)

  # Create IRanges object for the full amino acid sequence
  aa_range        <- IRanges(start = 1, end = prot_length)
  names(aa_range) <- pid

  # Map amino acid coordinates → genomic coordinates
  mapped_coords <- proteinToGenome(aa_range, edb)

  # Store as data frame
  mapping_results[[pid]] <- as.data.frame(mapped_coords[[1]])
}

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000437799'. The returned genomic coordinates might thus not be correct for this protein."
0/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000397299'. The returned genomic coordinates might thus not be correct for this protein."
0/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000412754'. The returned genomic coordinates might thus not be correct for this protein.

In [19]:
# ---------------------------------------------------------
# 7. COMBINE AND SAVE
# ---------------------------------------------------------
final_map_df <- do.call(rbind, mapping_results)

output_filename <- paste0(my_gene, "_protein_to_genome_map.tsv")

write.table(final_map_df, file = output_filename,
            sep = "\t", row.names = FALSE, quote = FALSE)

cat("\n--- Success! ---\n")
cat("Mapping complete! File saved as:", output_filename, "\n")


--- Success! ---
Mapping complete! File saved as: CYP2E1_protein_to_genome_map.tsv 
